Voorkeur mixed vs dames ipv mixed tegen heren - TBD
geen twee teams 2x tegen elkaar - gedaan
hoofdrol kan niet tegen lol
Doorcombineren tot elke 
voorkeur ronden verplaatsen naar ronde 2

In [25]:
from __future__ import annotations
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict, Set, Optional
from collections import defaultdict
import random

@dataclass(frozen=True)
class Team:
    niveau: int         # 1=Hoofdrol, 2=Bijrol, 3=Op de set
    geslacht: str       # "Heren" | "Dames" | "Mixed"
    naam: str
    leeftijd: str       # Niet gebruikt voor planning;

@dataclass
class Match:
    ronde: int
    veld: int
    team_a: Team
    team_b: Team


# Verplicht aantal wedstrijden per niveau; Op de set heeft 2 verplicht + 1 optioneel
VERPLICHTE_WEDSTRIJDEN = {1: 3, 2: 3, 3: 2}
OPTIONELE_WEDSTRIJDEN = {1: 0, 2: 0, 3: 1}  # 1 extra mogelijk voor "Op de set" als er plek is

gespeelde_koppels = set()


def lees_teams(pad:str, sheet_name:str = "data") -> list[Team]:\

    teams_df = pd.read_excel(pad, sheet_name=sheet_name)
    teams = []
    for _, row in teams_df.iterrows():
        teams.append(Team(
            niveau = _niveau_map(row["Nivo"]),
            geslacht = row["Dames/Heren"],
            naam = row["Naam"],
            leeftijd = row["Jong/oud"]
        ))

    return teams

def _niveau_map(niveau:str) -> int:

    data = {
        "Hoofdrol": 1,
        "Bijrol": 2,
        "Op de set": 3,
    }

    return data.get(niveau)

def _leeftijd_map(niveau:str) -> int:

    data = {
        "Jong": 1,
        "Midden": 2,
        "Oud": 3,
    }

    return data.get(niveau)

def _naam_index(teams: List[Team]) -> Dict[str, Team]:
    ix = {}
    for t in teams:
        if t.naam in ix:
            raise ValueError(f"Teamnaam '{t.naam}' komt dubbel voor; namen moeten uniek zijn.")
        ix[t.naam] = t
    return ix

def _consecutieve_rondes_toegestaan(vorige: Optional[int], huidige: int) -> bool:
    """Teams mogen niet twee rondes achter elkaar spelen, behalve 5 -> 6 (pauze ertussen)."""
    if vorige is None:
        return True
    if huidige == vorige + 1 and not (vorige == 5 and huidige == 6):
        return False
    return True

def _geslacht_compatibel(a: Team, b: Team) -> bool:
    """Mixed kan met iedereen; anders bij voorkeur zelfde geslacht."""
    #if a.geslacht == "Mixed" or b.geslacht == "Mixed":
    #    return True
    
    if a.geslacht == "Mixed" and b.geslacht == "Dames":
        return True

    if b.geslacht == "Mixed" and a.geslacht == "Dames":
        return True
        
    return a.geslacht == b.geslacht

def _niveau_distance(a: Team, b: Team) -> int:
    return abs(a.niveau - b.niveau)

def _compatibiliteit_score(a: Team, b: Team) -> int:
    """
    Hogere score = betere match (zonder geslachtscomponent).
    - zelfde niveau sterk prefereren
    - kleine niveau-afstand toelaten als nodig
    Let op: ongeldige geslachtscombinaties worden al gefilterd en komen hier niet doorheen.
    """
    score = 100
    score -= abs(a.niveau - b.niveau) * 20  # 0, 10, 20
    score -= abs(_leeftijd_map(a.leeftijd) - _leeftijd_map(b.leeftijd)) * 10
    # Optioneel: lichte 'anti Mixed-Mixed' bias weg of laten staan; hier verwijderen we die ook:
    return score

def _pareer_gretig(beschikbaar: List[Team],
                   al_gespeeld: Set[frozenset],
                   ronde: int) -> List[Tuple[Team, Team]]:
    """
    Greedy pairing op basis van compatibiliteit. Vermijdt herhalingsduels waar mogelijk.
    """
    paren: List[Tuple[Team, Team]] = []
    pool = beschikbaar[:]  # kopie
    # Sorteer: eerst hogere niveaus, dan D/H boven Mixed voor scherpere matches
    pool.sort(key=lambda t: (t.niveau, t.geslacht != "Mixed", t.naam))
    gebruikt: Set[Team] = set()

    for i, t in enumerate(pool):
        if t in gebruikt:
            continue
        # Kandidaten die nog niet gebruikt zijn
        #kandidaten = [k for k in pool[i+1:] if k not in gebruikt]
        
        
        kandidaten = [
            k for k in pool[i+1:]
            if k not in gebruikt
            and _geslacht_compatibel(t, k)
            and frozenset((t.naam, k.naam)) not in al_gespeeld
        ]


        # Scoreer kandidaten, vermijd herhalingen
        kandidaten.sort(key=lambda k: (
            (frozenset((t.naam, k.naam)) in al_gespeeld),  # False (0) is beter
            -_compatibiliteit_score(t, k),
            k.naam
        ))
        if not kandidaten:
            continue
        k = kandidaten[0]
        paren.append((t, k))
        gebruikt.add(t)
        gebruikt.add(k)

    return paren

def genereer_schema(
    teams: List[Team],
    voorkeuren: List[Tuple[str, str]],
    n_rondes: int = 7,
    n_velden: int = 12,
    seed: Optional[int] = 42
) -> Tuple[List[Match], Dict[str, int], Dict[str, int]]:
    """
    Returned:
      - lijst met Match
      - resterende_verplicht[name] (0 betekent voldaan)
      - resterende_optioneel[name] (0 betekent geen optionele plek meer)
    """
    if seed is not None:
        random.seed(seed)

    naam2team = _naam_index(teams)

    # Init counters
    resterend_verplicht: Dict[Team, int] = {
        t: VERPLICHTE_WEDSTRIJDEN.get(t.niveau, 0) for t in teams
    }
    resterend_opt: Dict[Team, int] = {
        t: OPTIONELE_WEDSTRIJDEN.get(t.niveau, 0) for t in teams
    }
    laatste_ronde: Dict[Team, Optional[int]] = {t: None for t in teams}

    # Normaliseer voorkeuren: filter alleen bekende teams en maak als set van frozensets
    voorkeur_set: Set[frozenset] = set()
    for a, b in voorkeuren:
        if a in naam2team and b in naam2team and a != b:
            voorkeur_set.add(frozenset((a, b)))

    ongeplande_voorkeuren = set(voorkeur_set)

    # Verdeel voorkeuren vooraf over rondes zodat ze niet allemaal vooraan landen
    voorkeur_lijst = list(voorkeur_set)
    random.shuffle(voorkeur_lijst)

    voorkeur_doelronde: Dict[frozenset, int] = {}
    for i, pair in enumerate(voorkeur_lijst):
        voorkeur_doelronde[pair] = (i % n_rondes) + 1

    # Voor trace: welke paren hebben al tegen elkaar gespeeld?
    al_gespeeld: Set[frozenset] = set()

    wedstrijden: List[Match] = []

    # Helper om beschikbaarheidslijst voor ronde op te bouwen
    def _beschikbare_teams(ronde: int, alleen_verplicht: bool) -> List[Team]:
        res = []
        for t in teams:
            if not _consecutieve_rondes_toegestaan(laatste_ronde[t], ronde):
                continue

            # Mag maximaal 1 keer per ronde
            if any(m.ronde == ronde and (m.team_a == t or m.team_b == t) for m in wedstrijden):
                continue

            # Alleen teams met resterende capaciteit
            if alleen_verplicht:
                if resterend_verplicht[t] > 0:
                    res.append(t)
            else:
                # Eerst teams die nog verplicht moeten,
                # daarna (bij overcapaciteit) optionele voor "Op de set"
                if resterend_verplicht[t] > 0 or resterend_opt[t] > 0:
                    res.append(t)

        return res

    # Plan per ronde
    for ronde in range(1, n_rondes + 1):
        veld_teller = 1

        # 1) VOORKEUREN GESPREID OVER DE RONDES
        geplande_deze_ronde: Set[Team] = set()

        resterende_rondes = n_rondes - ronde + 1
        resterende_voorkeuren = len(ongeplande_voorkeuren)

        # Verdeel voorkeuren ongeveer gelijkmatig over resterende rondes
        voorkeur_quota = min(
            n_velden,
            math.ceil(resterende_voorkeuren / resterende_rondes) if resterende_voorkeuren > 0 else 0
        )
        geplande_voorkeuren_deze_ronde = 0

        def _dringendheidskey(fs: frozenset) -> Tuple[int, int, int, int, str]:
            a_name, b_name = list(fs)
            a, b = naam2team[a_name], naam2team[b_name]

            doelronde = voorkeur_doelronde.get(fs, ronde)

            # Eerst voorkeuren die "aan de beurt" zijn of al te laat zijn
            overdue_flag = 0 if doelronde <= ronde else 1

            return (
                overdue_flag,
                abs(doelronde - ronde),
                -(resterend_verplicht[a] + resterend_verplicht[b]),
                -_compatibiliteit_score(a, b),
                f"{a_name}-{b_name}"
            )

        for pair in sorted(list(ongeplande_voorkeuren), key=_dringendheidskey):
            if veld_teller > n_velden:
                break
            if geplande_voorkeuren_deze_ronde >= voorkeur_quota:
                break

            a_name, b_name = list(pair)
            a, b = naam2team[a_name], naam2team[b_name]

            # Beide beschikbaar en nog iets te spelen?
            if a in geplande_deze_ronde or b in geplande_deze_ronde:
                continue
            if not _consecutieve_rondes_toegestaan(laatste_ronde[a], ronde):
                continue
            if not _consecutieve_rondes_toegestaan(laatste_ronde[b], ronde):
                continue
            if resterend_verplicht[a] == 0 and resterend_opt[a] == 0:
                continue
            if resterend_verplicht[b] == 0 and resterend_opt[b] == 0:
                continue

            # Als je geslachtscompatibiliteit wilt afdwingen, zet dit aan:
            # if not _geslacht_compatibel(a, b):
            #     continue

            # Plan de voorkeursmatch
            wedstrijden.append(Match(ronde, veld_teller, a, b))
            veld_teller += 1
            geplande_voorkeuren_deze_ronde += 1

            geplande_deze_ronde.update([a, b])
            laatste_ronde[a] = ronde
            laatste_ronde[b] = ronde
            al_gespeeld.add(frozenset((a.naam, b.naam)))

            # Consumeer verplicht eerst; als op is, consumeer optioneel
            for t in (a, b):
                if resterend_verplicht[t] > 0:
                    resterend_verplicht[t] -= 1
                elif resterend_opt[t] > 0:
                    resterend_opt[t] -= 1

            ongeplande_voorkeuren.discard(pair)

        # 2) VUL MET "GEWONE" MATCHES (eerst verplicht, daarna optioneel bij ruimte)
        for fase in ("verplicht", "opt"):
            if veld_teller > n_velden:
                break

            alleen_verplicht = (fase == "verplicht")
            beschikbaar = [
                t for t in _beschikbare_teams(ronde, alleen_verplicht)
                if t not in geplande_deze_ronde
            ]

            # Greedy paring op basis van compatibiliteit en vermijden van herhalingen
            paren = _pareer_gretig(beschikbaar, al_gespeeld, ronde)

            for a, b in paren:
                if veld_teller > n_velden:
                    break

                # Extra check: beide hebben capaciteit in deze fase
                if alleen_verplicht:
                    if resterend_verplicht[a] == 0 or resterend_verplicht[b] == 0:
                        continue
                else:
                    if resterend_verplicht[a] == 0 and resterend_opt[a] == 0:
                        continue
                    if resterend_verplicht[b] == 0 and resterend_opt[b] == 0:
                        continue

                # Plan
                wedstrijden.append(Match(ronde, veld_teller, a, b))
                veld_teller += 1
                geplande_deze_ronde.update([a, b])
                laatste_ronde[a] = ronde
                laatste_ronde[b] = ronde
                al_gespeeld.add(frozenset((a.naam, b.naam)))

                # Consumeer capaciteit
                for t in (a, b):
                    if resterend_verplicht[t] > 0:
                        resterend_verplicht[t] -= 1
                    elif resterend_opt[t] > 0:
                        resterend_opt[t] -= 1

    # Maak nette, naamgebaseerde resterend-dicts voor rapportage
    rest_verplicht_by_name = {t.naam: resterend_verplicht[t] for t in teams}
    rest_opt_by_name = {t.naam: resterend_opt[t] for t in teams}

    return wedstrijden, rest_verplicht_by_name, rest_opt_by_name

def print_schema(wedstrijden: List[Match]) -> None:
    if not wedstrijden:
        print("Geen wedstrijden ingepland.")
        return
    wedstrijden_sorted = sorted(wedstrijden, key=lambda m: (m.ronde, m.veld))
    ronde = None
    for m in wedstrijden_sorted:
        if ronde != m.ronde:
            ronde = m.ronde
            print(f"\n=== Ronde {ronde} ===")
        print(80*'+')
        print(f"Veld {m.veld:02d}: {m.team_a.naam}  vs  {m.team_b.naam}")
        print(f"{m.team_a.geslacht} -- {m.team_b.geslacht}")
        print(f"{m.team_a.leeftijd} -- {m.team_b.leeftijd}")
        print(f"{m.team_a.niveau} -- {m.team_b.niveau}")



In [ ]:
def bereken_minimaal_aantal_velden(teams, voorkeuren, n_rondes):
    for i in range(1, 15):
        
        schema, rest_verplicht, rest_optioneel = genereer_schema(teams, voorkeuren, n_rondes=n_rondes, n_velden=i, seed=7)
        if not any(rest_verplicht.values()):
            print(rest_optioneel)
            return schema, i
    
    return None, -1

In [12]:
voorkeuren = [
    ("Wiegert Schut", "Sebastiaan Schroder"),
    #("Wiegert Schut", "Robert Jan Prins"),
    ("Sebastiaan Schroder", "Jan Lodewijk Smit"),
    ("Chrisje Schroder", "Nanette Wielenga"),
    ]

teams = lees_teams("Indeling teams.xlsx")


In [13]:
import math

In [26]:
schema, _, _ = genereer_schema(teams, voorkeuren, n_rondes=7, n_velden=12, seed=7)

print_schema(schema)


=== Ronde 1 ===
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Veld 01: Sebastiaan Schroder  vs  Jan Lodewijk Smit
Heren -- Heren
Jong -- Oud
2 -- 3
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Veld 02: Wiegert Schut  vs  Sabrina Everard
Mixed -- Dames
Jong -- Jong
1 -- 1
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Veld 03: Anna van Gellicum  vs  Cathelijne Roukema
Dames -- Dames
Oud -- Oud
1 -- 1
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Veld 04: Bram Blees  vs  Olivier Buitink
Heren -- Heren
Jong -- Jong
1 -- 1
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Veld 05: Sascha de Lanoy Meijer  vs  Suzanne Kooy-Olijslagers
Dames -- Mixed
Oud -- Oud
1 -- 2
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Veld 06: Thijs Parlevliet  vs  Yaniek Buitink
Heren -- Heren
Jong -- Jong
1 -

In [27]:
schema, velden = bereken_minimaal_aantal_velden(teams, voorkeuren, n_rondes=7)
print_schema(schema)

{'Chrisje Schroder': 0, 'Evie Buitink': 0, 'Sabrina Everard': 0, 'Jochem van Pelt': 0, 'Sarah Claproth': 0, 'Tessel van Well': 0, 'Wiegert Schut': 0, 'Anna van Gellicum': 0, 'Cathelijne Roukema': 0, 'Eva Kernkamp': 0, 'Nanette Wielenga': 1, 'Sascha de Lanoy Meijer': 0, 'Suzanne Kooy-Olijslagers': 0, 'Bas Viëtor': 0, 'Bram Blees': 0, 'Olivier Buitink': 0, 'Sebastiaan Schroder': 0, 'Thijs Parlevliet': 0, 'Yaniek Buitink': 0, 'Cappie de Muralt': 0, 'Hessel de Jong': 1, 'Jan Lodewijk Smit': 0, 'Pieter van Aken': 0, 'Rob van Well': 0, 'Robert Jan Prins': 1, 'Rogier Krabbendam': 0, 'Dames1': 0}

=== Ronde 1 ===
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Veld 01: Sebastiaan Schroder  vs  Jan Lodewijk Smit
Heren -- Heren
Jong -- Oud
2 -- 3
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Veld 02: Wiegert Schut  vs  Sabrina Everard
Mixed -- Dames
Jong -- Jong
1 -- 1
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++